# Predicting Heart Disease - S6E2
## v4: Feature Growth + Target/Freq Encoding + Tuned CatBoost + Residual RF

公開Notebook (0.95382) のアプローチと我々の手法を統合:
- Target Encoding + Frequency Encoding（全カラム）
- Feature Growth: TE特徴量の相関ベースで交互作用を自動選択
- CatBoost: 強い正則化パラメータ
- Rank Stack + Residual Random Forest
- 原データ追加（我々のオリジナル）

In [1]:
import numpy as np
import pandas as pd
import gc
import warnings
from itertools import combinations

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

warnings.filterwarnings('ignore')

In [2]:
class CFG:
    seed = 42
    n_splits = 5
    target_col = 'Heart Disease'
    use_original = True
    n_top_pairs = 8  # Feature Growthで使う交互作用ペア数

class Paths:
    p = '/Users/shirokoshikentaro/Desktop/Predicting Heart Disease/Data'
    train = p + '/train.csv'
    test = p + '/test.csv'
    sample = p + '/sample_submission.csv'
    original = p + '/dataset_heart.csv'

## 1. Data Loading + 原データ追加

In [3]:
train = pd.read_csv(Paths.train)
test = pd.read_csv(Paths.test)
sample_submission = pd.read_csv(Paths.sample)

print(f'Train shape (synthetic): {train.shape}')
print(f'Test shape: {test.shape}')

# 原データ追加
if CFG.use_original:
    original = pd.read_csv(Paths.original)
    rename_map = {
        'age': 'Age',
        'sex ': 'Sex',
        'chest pain type': 'Chest pain type',
        'resting blood pressure': 'BP',
        'serum cholestoral': 'Cholesterol',
        'fasting blood sugar': 'FBS over 120',
        'resting electrocardiographic results': 'EKG results',
        'max heart rate': 'Max HR',
        'exercise induced angina': 'Exercise angina',
        'oldpeak': 'ST depression',
        'ST segment': 'Slope of ST',
        'major vessels': 'Number of vessels fluro',
        'thal': 'Thallium',
        'heart disease': 'Heart Disease',
    }
    original = original.rename(columns=rename_map)
    original['Heart Disease'] = original['Heart Disease'].map({1: 0, 2: 1})

    if 'id' not in original.columns:
        max_id = train['id'].max()
        original['id'] = range(max_id + 1, max_id + 1 + len(original))
    original = original[train.columns]

    train = pd.concat([train, original], axis=0, ignore_index=True)
    cols_for_dedup = [c for c in train.columns if c != 'id']
    n_before = len(train)
    train = train.drop_duplicates(subset=cols_for_dedup, keep='first').reset_index(drop=True)
    print(f'After concat + dedup: {len(train)} (+{len(original)} orig, -{n_before - len(train)} dups)')

Train shape (synthetic): (630000, 15)
Test shape: (270000, 14)
After concat + dedup: 630270 (+270 orig, -0 dups)


## 2. Target Encoding + Column Setup

In [4]:
# ターゲット変換
TARGET = CFG.target_col
if train[TARGET].dtype == 'object' or train[TARGET].dtype.name == 'category':
    train[TARGET] = train[TARGET].map({'Absence': 0, 'Presence': 1})

y = train[TARGET].values.astype(np.uint8)
test_ids = test['id']

print(f'Target dtype: {train[TARGET].dtype}')
print(f'Target unique: {np.unique(y)}')
print(f'Target distribution: 0={np.mean(y==0):.3f}, 1={np.mean(y==1):.3f}')
assert np.isnan(y).sum() == 0, 'ERROR: NaN in target!'

# 生の特徴量
X_tr_raw = train.drop(columns=[TARGET, 'id'])
X_te_raw = test.drop(columns=['id'])

cat_cols = [
    'Sex', 'Chest pain type', 'FBS over 120', 'EKG results',
    'Exercise angina', 'Slope of ST',
    'Number of vessels fluro', 'Thallium'
]
num_cols = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']
all_base_cols = cat_cols + num_cols

print(f'\nCategorical ({len(cat_cols)}): {cat_cols}')
print(f'Numerical ({len(num_cols)}): {num_cols}')

Target dtype: float64
Target unique: [0 1]
Target distribution: 0=0.552, 1=0.448

Categorical (8): ['Sex', 'Chest pain type', 'FBS over 120', 'EKG results', 'Exercise angina', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
Numerical (5): ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']


## 3. Frequency Encoding

In [5]:
def freq_encode(tr, te, cols):
    """train+test全体での出現頻度を特徴量にする"""
    all_df = pd.concat([tr, te])
    tr_out = pd.DataFrame(index=tr.index)
    te_out = pd.DataFrame(index=te.index)
    for c in cols:
        freq = all_df[c].value_counts(normalize=True)
        tr_out[c + '_freq'] = tr[c].map(freq)
        te_out[c + '_freq'] = te[c].map(freq)
    return tr_out, te_out

tr_freq, te_freq = freq_encode(X_tr_raw, X_te_raw, all_base_cols)
print(f'Frequency features: {tr_freq.shape[1]} columns')
print(f'Columns: {tr_freq.columns.tolist()}')

Frequency features: 13 columns
Columns: ['Sex_freq', 'Chest pain type_freq', 'FBS over 120_freq', 'EKG results_freq', 'Exercise angina_freq', 'Slope of ST_freq', 'Number of vessels fluro_freq', 'Thallium_freq', 'Age_freq', 'BP_freq', 'Cholesterol_freq', 'Max HR_freq', 'ST depression_freq']


## 4. Target Encoding (K-Fold Safe)

In [6]:
skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
tr_te = pd.DataFrame(index=X_tr_raw.index)
te_te = pd.DataFrame(index=X_te_raw.index)

for c in all_base_cols:
    tr_te[c + '_te'] = 0.0
    for tr_i, val_i in skf_te.split(X_tr_raw, y):
        means = train.iloc[tr_i].groupby(c)[TARGET].mean()
        tr_te.iloc[val_i, tr_te.columns.get_loc(c + '_te')] = X_tr_raw.iloc[val_i][c].map(means)
    te_te[c + '_te'] = X_te_raw[c].map(train.groupby(c)[TARGET].mean())

print(f'Target Encoding features: {tr_te.shape[1]} columns')
print(f'Columns: {tr_te.columns.tolist()}')

Target Encoding features: 13 columns
Columns: ['Sex_te', 'Chest pain type_te', 'FBS over 120_te', 'EKG results_te', 'Exercise angina_te', 'Slope of ST_te', 'Number of vessels fluro_te', 'Thallium_te', 'Age_te', 'BP_te', 'Cholesterol_te', 'Max HR_te', 'ST depression_te']


## 5. Feature Growth (Apriori-lite)
TE特徴量同士の相関が高いペアを自動選出し、交互作用を追加

In [7]:
base = tr_te.fillna(0)
corr_scores = {}

for a, b in combinations(base.columns, 2):
    corr_scores[(a, b)] = abs(np.corrcoef(base[a], base[b])[0, 1])

top_pairs = sorted(corr_scores, key=corr_scores.get, reverse=True)[:CFG.n_top_pairs]

print(f'Top {CFG.n_top_pairs} correlated TE pairs:')
for a, b in top_pairs:
    print(f'  {a} × {b} (corr={corr_scores[(a,b)]:.4f})')
    tr_te[f'{a}_x_{b}'] = tr_te[a] * tr_te[b]
    te_te[f'{a}_x_{b}'] = te_te[a] * te_te[b]

print(f'\nTE features after Feature Growth: {tr_te.shape[1]} columns')

Top 8 correlated TE pairs:
  Slope of ST_te × ST depression_te (corr=0.4691)
  Chest pain type_te × Thallium_te (corr=0.3636)
  Exercise angina_te × Thallium_te (corr=0.3570)
  Number of vessels fluro_te × Thallium_te (corr=0.3530)
  Thallium_te × Max HR_te (corr=0.3349)
  Thallium_te × ST depression_te (corr=0.3306)
  Number of vessels fluro_te × ST depression_te (corr=0.3119)
  Slope of ST_te × Thallium_te (corr=0.3106)

TE features after Feature Growth: 21 columns


## 6. 手動交互作用特徴量（我々のオリジナル）

In [8]:
def add_manual_features(df):
    """ドメイン知識ベースの交互作用特徴量"""
    out = pd.DataFrame(index=df.index)

    # 型変換
    for col in df.select_dtypes(include=['int8', 'int16']).columns:
        df[col] = df[col].astype('int32')
    for col in df.select_dtypes(include=['float16']).columns:
        df[col] = df[col].astype('float32')

    # 数値 x 数値
    out['Age_x_MaxHR'] = df['Age'] * df['Max HR']
    out['Age_div_MaxHR'] = df['Age'] / (df['Max HR'] + 1e-5)
    out['STdep_x_MaxHR'] = df['ST depression'] * df['Max HR']
    out['BP_x_Chol'] = df['BP'] * df['Cholesterol']
    out['Age_x_STdep'] = df['Age'] * df['ST depression']
    out['Age_x_Vessels'] = df['Age'] * df['Number of vessels fluro']
    out['BP_x_MaxHR'] = df['BP'] * df['Max HR']
    out['Chol_x_Age'] = df['Cholesterol'] * df['Age']
    out['STdep_x_Vessels'] = df['ST depression'] * df['Number of vessels fluro']

    # 数値 x カテゴリ
    out['Sex_x_MaxHR'] = df['Sex'] * df['Max HR']
    out['Sex_x_Chol'] = df['Sex'] * df['Cholesterol']
    out['ChestPain_x_STdep'] = df['Chest pain type'] * df['ST depression']
    out['ChestPain_x_MaxHR'] = df['Chest pain type'] * df['Max HR']
    out['ExAngina_x_STdep'] = df['Exercise angina'] * df['ST depression']
    out['ExAngina_x_MaxHR'] = df['Exercise angina'] * df['Max HR']
    out['Thallium_x_Vessels'] = df['Thallium'] * df['Number of vessels fluro']
    out['SlopeSTx_STdep'] = df['Slope of ST'] * df['ST depression']

    # 差分・比率
    out['MaxHR_reserve'] = (220 - df['Age']) - df['Max HR']
    out['MaxHR_ratio'] = df['Max HR'] / (220 - df['Age'] + 1e-5)

    return out

tr_manual = add_manual_features(X_tr_raw)
te_manual = add_manual_features(X_te_raw)
print(f'Manual interaction features: {tr_manual.shape[1]} columns')

Manual interaction features: 19 columns


## 7. Final Feature Set

In [9]:
# 全特徴量を結合: 生データ + Freq Encoding + Target Encoding + Feature Growth + 手動交互作用
X_train = pd.concat([X_tr_raw, tr_freq, tr_te, tr_manual], axis=1).fillna(0)
X_test = pd.concat([X_te_raw, te_freq, te_te, te_manual], axis=1).fillna(0)

print(f'Final feature set:')
print(f'  X_train: {X_train.shape}')
print(f'  X_test:  {X_test.shape}')
print(f'\nBreakdown:')
print(f'  Raw features:        {X_tr_raw.shape[1]}')
print(f'  Frequency Encoding:  {tr_freq.shape[1]}')
print(f'  Target Encoding:     {tr_te.shape[1]}')
print(f'  Manual interactions: {tr_manual.shape[1]}')
print(f'  Total:               {X_train.shape[1]}')

Final feature set:
  X_train: (630270, 66)
  X_test:  (270000, 66)

Breakdown:
  Raw features:        13
  Frequency Encoding:  13
  Target Encoding:     21
  Manual interactions: 19
  Total:               66


## 8. Model Parameters
公開Notebook (0.95382) ベースの強正則化パラメータ

In [10]:
cat_params = {
    'iterations': 3000,
    'learning_rate': 0.03,
    'depth': 4,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'auto_class_weights': 'Balanced',
    'subsample': 0.65,
    'l2_leaf_reg': 12,
    'random_strength': 1.2,
    'bootstrap_type': 'Bernoulli',
    'task_type': 'CPU',       # GPUがなければCPU
    'verbose': False,
}

xgb_params = {
    'n_estimators': 2500,
    'learning_rate': 0.03,
    'max_depth': 4,
    'subsample': 0.65,
    'colsample_bytree': 0.8,
    'reg_lambda': 12,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'verbosity': 0,
}

rf_params = {
    'n_estimators': 400,
    'max_depth': 6,
    'min_samples_leaf': 50,
    'class_weight': 'balanced',
    'n_jobs': -1,
    'random_state': CFG.seed,
}

print('Model parameters configured!')
print(f'  CatBoost: depth={cat_params["depth"]}, lr={cat_params["learning_rate"]}, l2={cat_params["l2_leaf_reg"]}')
print(f'  XGBoost:  depth={xgb_params["max_depth"]}, lr={xgb_params["learning_rate"]}, lambda={xgb_params["reg_lambda"]}')
print(f'  RF:       depth={rf_params["max_depth"]}, n_est={rf_params["n_estimators"]}')

Model parameters configured!
  CatBoost: depth=4, lr=0.03, l2=12
  XGBoost:  depth=4, lr=0.03, lambda=12
  RF:       depth=6, n_est=400


## 9. 5-Fold Training + Rank Stack

In [11]:
skf = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)

# CatBoost + XGBoost 個別のOOF/test予測も保存
oof_cb = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))
oof_rank = np.zeros(len(X_train))

test_cb = np.zeros(len(X_test))
test_xgb_pred = np.zeros(len(X_test))
test_rank = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
    print(f'\n===== Fold {fold + 1} =====')

    # --- CatBoost ---
    cb = CatBoostClassifier(**cat_params, random_seed=CFG.seed + fold)
    cb.fit(X_train.iloc[tr_idx], y[tr_idx])
    val_cb = cb.predict_proba(X_train.iloc[val_idx])[:, 1]
    te_cb = cb.predict_proba(X_test)[:, 1]
    oof_cb[val_idx] = val_cb
    test_cb += te_cb / CFG.n_splits
    print(f'  CatBoost AUC: {roc_auc_score(y[val_idx], val_cb):.6f}')

    # --- XGBoost ---
    xgb_model = XGBClassifier(**xgb_params, random_state=CFG.seed + fold)
    xgb_model.fit(X_train.iloc[tr_idx], y[tr_idx])
    val_xgb = xgb_model.predict_proba(X_train.iloc[val_idx])[:, 1]
    te_xgb = xgb_model.predict_proba(X_test)[:, 1]
    oof_xgb[val_idx] = val_xgb
    test_xgb_pred += te_xgb / CFG.n_splits
    print(f'  XGBoost  AUC: {roc_auc_score(y[val_idx], val_xgb):.6f}')

    # --- Fold内Rank Average ---
    val_rank = (rankdata(val_cb) + rankdata(val_xgb)) / 2 / len(val_cb)
    te_rank = (rankdata(te_cb) + rankdata(te_xgb)) / 2 / len(te_cb)
    oof_rank[val_idx] = val_rank
    test_rank += te_rank / CFG.n_splits
    print(f'  Rank Avg AUC: {roc_auc_score(y[val_idx], val_rank):.6f}')

    gc.collect()

print('\n' + '=' * 50)
print(f'CatBoost OOF AUC: {roc_auc_score(y, oof_cb):.6f}')
print(f'XGBoost  OOF AUC: {roc_auc_score(y, oof_xgb):.6f}')
print(f'Rank Avg OOF AUC: {roc_auc_score(y, oof_rank):.6f}')


===== Fold 1 =====
  CatBoost AUC: 0.955859
  XGBoost  AUC: 0.955550
  Rank Avg AUC: 0.955794

===== Fold 2 =====
  CatBoost AUC: 0.955666
  XGBoost  AUC: 0.955216
  Rank Avg AUC: 0.955554

===== Fold 3 =====
  CatBoost AUC: 0.955012
  XGBoost  AUC: 0.954566
  Rank Avg AUC: 0.954896

===== Fold 4 =====
  CatBoost AUC: 0.955124
  XGBoost  AUC: 0.954533
  Rank Avg AUC: 0.954948

===== Fold 5 =====
  CatBoost AUC: 0.955391
  XGBoost  AUC: 0.954478
  Rank Avg AUC: 0.955163

CatBoost OOF AUC: 0.955402
XGBoost  OOF AUC: 0.954822
Rank Avg OOF AUC: 0.955271


## 10. Residual Random Forest

In [12]:
rf = RandomForestClassifier(**rf_params)
rf.fit(oof_rank.reshape(-1, 1), y)

rf_oof_pred = rf.predict_proba(oof_rank.reshape(-1, 1))[:, 1]
rf_test_pred = rf.predict_proba(test_rank.reshape(-1, 1))[:, 1]

print(f'RF on rank OOF AUC: {roc_auc_score(y, rf_oof_pred):.6f}')

# Blend: Rank Stack + RF Residual
# 最適な混合比を探索
best_alpha = 0.85
best_auc_blend = 0

for alpha in np.arange(0.5, 1.0, 0.05):
    blended = alpha * oof_rank + (1 - alpha) * rf_oof_pred
    auc = roc_auc_score(y, blended)
    if auc > best_auc_blend:
        best_auc_blend = auc
        best_alpha = alpha

print(f'\nBest alpha (Rank weight): {best_alpha:.2f}')
print(f'Blended OOF AUC: {best_auc_blend:.6f}')

final_test = best_alpha * test_rank + (1 - best_alpha) * rf_test_pred

RF on rank OOF AUC: 0.955469

Best alpha (Rank weight): 0.50
Blended OOF AUC: 0.955368


## 11. Submission（複数パターン）

In [13]:
# メイン: Rank + RF Residual
sub_main = pd.DataFrame({'id': test_ids, TARGET: final_test.astype(float)})
sub_main.to_csv('submission_v4_rank_rf.csv', index=False)
print('✅ submission_v4_rank_rf.csv saved!')

# CatBoost単体
sub_cb = pd.DataFrame({'id': test_ids, TARGET: test_cb.astype(float)})
sub_cb.to_csv('submission_v4_catboost.csv', index=False)
print('✅ submission_v4_catboost.csv saved!')

# Rank Stack のみ（RF Residualなし）
sub_rank = pd.DataFrame({'id': test_ids, TARGET: test_rank.astype(float)})
sub_rank.to_csv('submission_v4_rank_only.csv', index=False)
print('✅ submission_v4_rank_only.csv saved!')

✅ submission_v4_rank_rf.csv saved!
✅ submission_v4_catboost.csv saved!
✅ submission_v4_rank_only.csv saved!


In [14]:
print('=' * 60)
print('FINAL SUMMARY (v4)')
print('=' * 60)
print(f'  Original data     : {"ON" if CFG.use_original else "OFF"}')
print(f'  Features          : {X_train.shape[1]} columns')
print(f'  Feature Growth    : top {CFG.n_top_pairs} TE pairs')
print(f'  CatBoost OOF AUC : {roc_auc_score(y, oof_cb):.6f}')
print(f'  XGBoost  OOF AUC : {roc_auc_score(y, oof_xgb):.6f}')
print(f'  Rank Avg OOF AUC : {roc_auc_score(y, oof_rank):.6f}')
print(f'  +RF Blend OOF AUC: {best_auc_blend:.6f}')
print(f'  RF blend alpha    : {best_alpha:.2f}')
print('=' * 60)
print('\nSubmission files:')
print('  1. submission_v5_rank_rf.csv   (メイン: Rank+RF)')
print('  2. submission_v5_catboost.csv  (CatBoost単体)')
print('  3. submission_v5_rank_only.csv (Rank Stackのみ)')
print('\n→ 3つとも提出してLBを比較してください')

FINAL SUMMARY (v4)
  Original data     : ON
  Features          : 66 columns
  Feature Growth    : top 8 TE pairs
  CatBoost OOF AUC : 0.955402
  XGBoost  OOF AUC : 0.954822
  Rank Avg OOF AUC : 0.955271
  +RF Blend OOF AUC: 0.955368
  RF blend alpha    : 0.50

Submission files:
  1. submission_v5_rank_rf.csv   (メイン: Rank+RF)
  2. submission_v5_catboost.csv  (CatBoost単体)
  3. submission_v5_rank_only.csv (Rank Stackのみ)

→ 3つとも提出してLBを比較してください


In [ ]:
# =============================================================================
# Public Blend
# =============================================================================
# 公開Notebookのsubmissionを読み込み
public_sub = pd.read_csv(Paths.p + '/submission.csv')
public_pred = public_sub[TARGET].values
my_pred = test_cb  # CatBoost単体（LB 0.95376）

# Rank Average で複数パターン作成
def rank_avg_2(pred1, pred2, w1):
    r1 = rankdata(pred1) / len(pred1)
    r2 = rankdata(pred2) / len(pred2)
    return w1 * r1 + (1 - w1) * r2

for w in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    blended = rank_avg_2(my_pred, public_pred, w)
    sub = pd.DataFrame({'id': test_ids, TARGET: blended.astype(float)})
    fname = f'submission_blend5_my{int(w*100)}_pub{int((1-w)*100)}.csv'
    sub.to_csv(fname, index=False)
    print(f'✅ {fname} saved!')

✅ submission_blend2_my10_pub90.csv saved!
✅ submission_blend2_my20_pub80.csv saved!
✅ submission_blend2_my30_pub70.csv saved!
✅ submission_blend2_my40_pub60.csv saved!
✅ submission_blend2_my50_pub50.csv saved!
✅ submission_blend2_my60_pub40.csv saved!
✅ submission_blend2_my70_pub30.csv saved!
